# Silver: clean, type, deduplicate, quarantine

Rebuilds the `silver_*` tables from bronze. Bad rows go to `silver_quarantine` with reasons, trailers to `silver_control_totals`, and one audit row per entity to `silver_load_audit`.

Requires the `env_portfolio` Environment (with the `portfolio_migration` wheel) and the `lh_portfolio` lakehouse attached as default. Logic and tests live in the GitHub repo; do not edit logic here.

In [ ]:
# Set by the data pipeline to @pipeline().RunId. Empty means a manual run.
batch_id = ""

In [ ]:
import json
import time
import uuid
from dataclasses import asdict

import portfolio_migration
from portfolio_migration.lakehouse import silver
from portfolio_migration.lakehouse.io import FabricLake

batch_id = batch_id or f"manual-{uuid.uuid4()}"
print(f"portfolio_migration {portfolio_migration.__version__}, batch {batch_id}")

started = time.perf_counter()
results = [asdict(r) for r in silver.run(FabricLake(spark), batch_id)]
seconds = round(time.perf_counter() - started, 1)
print(f"silver finished in {seconds} s")
display(spark.createDataFrame(results))

In [ ]:
# Returned to the pipeline as the activity output.
notebookutils.notebook.exit(json.dumps({"layer": "silver", "batch_id": batch_id, "seconds": seconds, "results": results}))